# Anomaly detection: MVTec AD на Kaggle

Перед запуском включите **Accelerator → GPU** и подключите Dataset с MVTec AD и DTD. Результаты сохраняются в `/kaggle/working/experiments`.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Включите GPU в настройках Notebook')
print(torch.cuda.get_device_name(0))

## Получение репозитория и установка зависимостей
При повторном открытии сессии выполните эти ячейки снова.

In [ ]:
%cd /kaggle/working
![ -d anomaly-detection-reproduction ] || git clone https://github.com/calmik0lya/anomaly-detection-reproduction.git
%cd /kaggle/working/anomaly-detection-reproduction
!git pull --ff-only
!pip install -q -r requirements.txt -r requirements-patchcore.txt -r requirements-anomalib.txt

## Восстановление результатов предыдущей сессии

После **Save Version** можно подключить output предыдущей версии как Dataset. Ячейка найдёт сохранённые experiment-папки в `/kaggle/input` и вернёт их в рабочую папку для `--resume`. При первом запуске она ничего не копирует.

In [ ]:
from pathlib import Path
import shutil
destination = Path('/kaggle/working/experiments')
destination.mkdir(parents=True, exist_ok=True)
restored = 0
for metrics_file in Path('/kaggle/input').glob('**/experiments/*__*__*/metrics.json'):
    source_run = metrics_file.parent
    target_run = destination / source_run.name
    if not target_run.exists():
        shutil.copytree(source_run, target_run)
        restored += 1
print('Восстановлено экспериментов:', restored)

## Проверка путей
Если имена подключённых Kaggle Dataset отличаются, измените две строки в `configs/paths/kaggle.yaml`.

In [ ]:
from pathlib import Path
import yaml
paths = yaml.safe_load(Path('configs/paths/kaggle.yaml').read_text())
for key in ('mvtec_path', 'dtd_images_path'):
    path = Path(paths[key])
    print(key, path, 'OK' if path.exists() else 'НЕ НАЙДЕН')
    if not path.exists():
        raise FileNotFoundError(f'Исправьте {key} в configs/paths/kaggle.yaml')

## Быстрая проверка команды без обучения

In [ ]:
!python run.py --config configs/models/patchcore.yaml --paths configs/paths/kaggle.yaml --category bottle --device cuda --dry-run

## Контрольный реальный запуск
Сначала запускаем только PatchCore на bottle и проверяем папку с конфигом, метриками и весами.

In [ ]:
!python run.py --config configs/models/patchcore.yaml --paths configs/paths/kaggle.yaml --category bottle --device cuda --output-dir /kaggle/working/experiments
!find /kaggle/working/experiments -maxdepth 3 -type f | sort

## Пакетный запуск с продолжением
Начинайте с одной-двух моделей. После завершения сохраните Version с output. В новой сессии подключите предыдущий output, выполните ячейку восстановления и повторите команду: `--resume` пропустит готовые эксперименты.

In [ ]:
!python sweep.py --models patchcore padim --paths configs/paths/kaggle.yaml --device cuda --output-dir /kaggle/working/experiments --resume --continue-on-error

Для следующих запусков замените модели на `stfpm`, затем `simplenet`, затем `draem`. DRAEM запускайте последним.